In [11]:
import pandas as pd

In [12]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2024.csv")

In [3]:
df.shape

(121605, 34)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121605 entries, 0 to 121604
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          121605 non-null  object 
 1   order_date              121605 non-null  object 
 2   customer_id             121605 non-null  object 
 3   product_id              121605 non-null  object 
 4   product_name            121605 non-null  object 
 5   category                121605 non-null  object 
 6   subcategory             121605 non-null  object 
 7   brand                   121605 non-null  object 
 8   original_price_inr      121605 non-null  object 
 9   discount_percent        121605 non-null  float64
 10  discounted_price_inr    121605 non-null  float64
 11  quantity                121605 non-null  int64  
 12  subtotal_inr            121605 non-null  float64
 13  delivery_charges        111877 non-null  float64
 14  final_amount_inr    

In [13]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [14]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [15]:
import pandas as pd

dfc['order_date'] = (
    dfc['order_date']
    .astype('string')
    .str.strip()
    .str.replace(r'[^\d/-]', '', regex=True)
)

dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    dayfirst=True,
    errors='coerce'
)

dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')


In [16]:
dfc['order_date'].head(50)

0     2024-03-01
1     2024-09-01
2     2024-09-01
3     2024-09-01
4            NaN
5            NaN
6     2024-12-01
7     2024-12-01
8            NaN
9            NaN
10           NaN
11           NaN
12           NaN
13           NaN
14    2024-06-01
15           NaN
16           NaN
17           NaN
18           NaN
19           NaN
20           NaN
21           NaN
22           NaN
23           NaN
24    2024-12-01
25           NaN
26           NaN
27           NaN
28           NaN
29           NaN
30           NaN
31           NaN
32           NaN
33           NaN
34           NaN
35           NaN
36    2024-08-01
37           NaN
38           NaN
39           NaN
40    2024-11-01
41    2024-02-01
42    2024-05-01
43           NaN
44           NaN
45    2024-11-01
46           NaN
47    2024-07-01
48           NaN
49           NaN
Name: order_date, dtype: object

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [18]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [24]:
import pandas as pd
import numpy as np
import re

# Example: if not already loaded
# dfc = pd.read_csv("amazon_india_2015_clean.csv")

def parse_rating(r):
    # 1. Handle missing values
    if pd.isna(r):
        return np.nan

    # 2. If already numeric (int or float), return it
    if isinstance(r, (int, float)):
        return float(r)

    # 3. Convert to string safely
    r = str(r).strip()

    # 4. Handle fraction ratings like "4/5"
    if '/' in r:
        try:
            a, b = r.split('/')
            return (float(a) / float(b)) * 5
        except:
            return np.nan

    # 5. Extract numeric part from strings like:
    #    "4.5 stars", "Rating: 3", "Rated 4 out of 5"
    match = re.search(r'\d+\.?\d*', r)
    if match:
        return float(match.group())

    # 6. Everything else → NaN
    return np.nan


# Apply cleaning
dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

# Optional but recommended: keep ratings between 1 and 5
dfc.loc[
    (dfc['customer_rating'] < 1) | (dfc['customer_rating'] > 5),
    'customer_rating'
] = np.nan

# Round to 2 decimal places
dfc['customer_rating'] = dfc['customer_rating'].round(2)


In [ ]:
dfc['customer_rating'].head(50)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [25]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [26]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [27]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [28]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [29]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [30]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [ ]:
print("Rows deleted:", (df_deduped))

In [19]:
len(dfc)

127132

In [32]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [33]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, price_outlier, price_corrected_flag]
Index: []

[0 rows x 36 columns]


In [34]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"       
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [35]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [36]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
    .astype(str)
    .str.replace('-', '', regex=False)
    .astype(float)
)


In [37]:
dfc["payment_method"].unique()

array(['UPI', 'Cash on Delivery', 'Debit Card', 'Credit Card', 'BNPL',
       'WALLET', 'Net Banking'], dtype=object)

In [38]:
import pandas as pd

columns = [
    "transaction_id","order_date","customer_id","product_id","product_name",
    "category","subcategory","brand","original_price_inr","discount_percent",
    "discounted_price_inr","quantity","subtotal_inr"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

transaction_id: ['TXN_2024_00000001', 'TXN_2024_00000002', 'TXN_2024_00000003', 'TXN_2024_00000004', 'TXN_2024_00000005', 'TXN_2024_00000006', 'TXN_2024_00000007', 'TXN_2024_00000008', 'TXN_2024_00000009', 'TXN_2024_00000010', 'TXN_2024_00000011', 'TXN_2024_00000012', 'TXN_2024_00000013', 'TXN_2024_00000014', 'TXN_2024_00000015', 'TXN_2024_00000016', 'TXN_2024_00000017', 'TXN_2024_00000018', 'TXN_2024_00000019', 'TXN_2024_00000020', 'TXN_2024_00000021', 'TXN_2024_00000022', 'TXN_2024_00000023', 'TXN_2024_00000024', 'TXN_2024_00000025', 'TXN_2024_00000026', 'TXN_2024_00000027', 'TXN_2024_00000028', 'TXN_2024_00000029', 'TXN_2024_00000030', 'TXN_2024_00000031', 'TXN_2024_00000032', 'TXN_2024_00000033', 'TXN_2024_00000034', 'TXN_2024_00000035', 'TXN_2024_00000036', 'TXN_2024_00000037', 'TXN_2024_00000038', 'TXN_2024_00000039', 'TXN_2024_00000040', 'TXN_2024_00000041', 'TXN_2024_00000042', 'TXN_2024_00000043', 'TXN_2024_00000044', 'TXN_2024_00000045', 'TXN_2

In [39]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0', '40.0']

final_amount_inr: ['10000.5', '10000.56', '100009.12', '100010.11', '10002.48', '10002.58', '10002.91', '100022.29', '100022.88', '100024.48', '10003.21', '10003.37', '100039.13', '100046.77', '100049.09', '100059.25', '100064.32', '100066.38', '10007.26', '10007.34', '100079.84', '100080.31', '100089.63', '10009.01', '10009.1', '10009.38', '10009.72', '10009.9', '100094.15', '100094.36', '10010.0', '100109.47', '10011.19', '10011.47', '100112.58', '100114.62', '100117.09', '10012.18', '10012.36', '100124.69', '100125.19', '100125.39', '10014.23', '100140.36', '100143.34', '100147.88999999998', '10015.24', '100151.12', '100152.66', '100152.83', '100155.18', '10016.85', '100165.32', '100168.23', '10017.75', '100173.87', '100178.54', '10018.2', '10020.18', '10021.84', '100211.42', '10022.55', '10022.77', '10022.85', '100224.68', '100225.21', '100228.83', '100229.06', '10023.76', '100230.78', '100235.28', '10024.13', '1002

In [40]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2024']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.09', '0.1', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.26', '0.27', '0.28', '0.29', '0.3', '0.31', '0.32', '0.33', '0.34', '0.35', '0.36', '0.37', '0.38', '0.39', '0.4', '0.41', '0.42', '0.43', '0.44', '0.45', '0.46', '0.47', '0.48', '0.49', '0.5', '0.51', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.6', '0.61', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68', '0.69', '0.7', '0.71', '0.72', '0.73', '0.75', '0.76', '0.77', '0.78', '1.2', '1.21', '1.24', '1.27', '1.29', '1.31', '1.32', '1.33', '1.37', '1.39', '1.4', '1.42', '1.46', '1.48',

In [41]:
print(len(dfc.columns))
print(dfc.columns.tolist())


34
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']


In [42]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


In [43]:
import pandas as pd

decimal_cols = dfc.select_dtypes(include=['float', 'float64']).columns

dfc[decimal_cols] = dfc[decimal_cols].round(2)
print(decimal_cols)


Index(['original_price_inr', 'discount_percent', 'discounted_price_inr',
       'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_rating', 'product_weight_kg', 'product_rating'],
      dtype='object')


In [45]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2024_clean.csv",header='infer',index=False)

In [44]:
dfc

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2024_00000001,2024-03-01,CUST_2024_00003447,PROD_000387,OnePlus OnePlus 6 64GB White,Electronics,Smartphones,OnePlus,39348.44,0.00,...,False,NaN,NaN,Delivered,1,2024,1,0.24,True,3.4
1,TXN_2024_00000002,2024-09-01,CUST_2018_00008611,PROD_001357,Nothing Phone (2a) Plus 64GB Blue,Electronics,Smartphones,Nothing,18694.97,0.00,...,False,NaN,5.0,Delivered,1,2024,1,0.15,True,4.1
2,TXN_2024_00000003,2024-09-01,CUST_2024_00002076,PROD_001899,Xiaomi Watch Premium,Electronics,Smart Watch,Xiaomi,59276.71,17.19,...,False,NaN,NaN,Delivered,1,2024,1,0.06,NaN,4.2
3,TXN_2024_00000004,2024-09-01,CUST_2024_00014530,PROD_000071,Xiaomi Redmi 2 32GB Black,Electronics,Smartphones,Xiaomi,32605.39,0.00,...,False,NaN,5.0,Delivered,1,2024,1,0.20,True,3.7
4,TXN_2024_00000005,NaN,CUST_2022_00043625,PROD_001144,OnePlus OnePlus 11R 256GB White,Electronics,Smartphones,OnePlus,72805.30,18.89,...,True,Republic Day Sale,4.5,Delivered,1,2024,1,0.21,True,3.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121600,TXN_2024_00021368_DUP,NaN,CUST_2024_00010041,PROD_000048,Samsung Galaxy J7 16GB White,Electronics,Smartphones,Samsung,27999.88,14.10,...,False,NaN,4.5,Delivered,3,2024,1,0.19,False,4.6
121601,TXN_2024_00030811_DUP,NaN,CUST_2024_00038545,PROD_000623,Oppo F11 64GB White,Electronics,Smartphones,Oppo,29256.30,0.00,...,False,NaN,4.0,Returned,4,2024,2,0.22,True,4.6
121602,TXN_2024_00005850_DUP,NaN,CUST_2016_00017631,PROD_001948,Noise Watch,Electronics,Smart Watch,Noise,33878.20,0.00,...,False,NaN,5.0,Delivered,1,2024,1,0.06,True,3.3
121603,TXN_2024_00005559_DUP,NaN,CUST_2024_00023947,PROD_001900,Xiaomi Watch Deluxe,Electronics,Smart Watch,Xiaomi,38806.15,27.19,...,False,NaN,3.0,Delivered,1,2024,1,0.08,True,3.5
